# Phase 2.5 — Model Interpretation

> **Diagnostic notebook — in-sample only.** One GBM is trained on the *full* dataset
> (all 6 panel symbols, no walk-forward) to expose what the model has learned.
> SHAP values, importances, and rolling accuracy reported here are **IS diagnostics**
> and must not be used to make OOS performance claims.
>
> For OOS performance numbers see `02_phase2_modeling.ipynb`.

**Sections**
1. [Feature importances](#1----feature-importances)
2. [SHAP summary](#2----shap-summary)
3. [SHAP dependence plots](#3----shap-dependence-plots)
4. [Prediction decomposition](#4----prediction-decomposition)
5. [Signal vs outcome scatter](#5----signal-vs-outcome-scatter)
6. [Rolling directional accuracy](#6----rolling-directional-accuracy)
7. [Confusion matrix](#7----confusion-matrix)
8. [Ridge coefficients](#8----ridge-coefficients)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
import xgboost as xgb
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

import quant.storage.catalog as catalog
from quant.config import settings
from quant.features.labels import generate_labels
from quant.features.engineering import build_features
from quant.models.gbm import GBMModel

PANEL_SYMS = ['AAPL', 'MSFT', 'JPM', 'JNJ', 'V', 'SPY']
HORIZON    = 5
INTERP_SYM = 'AAPL'  # symbol used for per-bar decompositions

print('Setup complete.')

In [ ]:
syms_sql = ', '.join(f"'{s}'" for s in PANEL_SYMS)
eq = catalog.query(f"""
    SELECT symbol, timestamp, open, high, low, adjClose AS close, volume
    FROM {catalog.table('equity_eod_tiingo')}
    WHERE symbol IN ({syms_sql})
    ORDER BY symbol, timestamp
""")
eq['timestamp'] = pd.to_datetime(eq['timestamp'])
eq = eq.set_index('timestamp')

prices_raw = {}
for sym in PANEL_SYMS:
    df = eq[eq['symbol'] == sym][['open', 'high', 'low', 'close', 'volume']].sort_index().dropna()
    prices_raw[sym] = df

features_raw = build_features(PANEL_SYMS, prices_raw)

features_by_sym = {}
labels_by_sym   = {}
for sym in PANEL_SYMS:
    feat = features_raw[sym]
    lab  = generate_labels(prices_raw[sym]['close'], horizon=HORIZON).series
    valid = feat.dropna().index.intersection(lab.dropna().index)
    features_by_sym[sym] = feat.loc[valid]
    labels_by_sym[sym]   = lab.loc[valid]

X_full = np.vstack([features_by_sym[s].values for s in PANEL_SYMS])
y_full = np.concatenate([labels_by_sym[s].values for s in PANEL_SYMS])
feature_names = list(features_by_sym['AAPL'].columns)

print(f'IS matrix: {X_full.shape}  ({len(PANEL_SYMS)} symbols)')
print(f'Features ({len(feature_names)}): {feature_names}')

In [ ]:
# WARNING: IN-SAMPLE ONLY — trained on full dataset without walk-forward.
# Use for interpretation only. Do not use to make OOS performance claims.
print('Training IS-only GBM (n_iter=50) on full panel...')
gbm_is = GBMModel(label_horizon=HORIZON, n_iter=50, n_splits=3, random_state=42)
gbm_is.fit(X_full, y_full)
print('Done.')

# Native XGBoost SHAP: shape (n_samples, n_features + 1), last col = bias
booster  = gbm_is._model.get_booster()
dm_full  = xgb.DMatrix(X_full, feature_names=feature_names)
shap_raw    = booster.predict(dm_full, pred_contribs=True)
shap_matrix = shap_raw[:, :-1]
shap_bias   = shap_raw[:, -1]

y_pred_full = gbm_is.predict(X_full)

# AAPL-only slice for per-bar sections
offset     = PANEL_SYMS.index(INTERP_SYM)
aapl_start = sum(len(features_by_sym[s]) for s in PANEL_SYMS[:offset])
n_aapl     = len(features_by_sym[INTERP_SYM])
aapl_slice = slice(aapl_start, aapl_start + n_aapl)

X_aapl    = X_full[aapl_slice]
y_aapl    = y_full[aapl_slice]
shap_aapl = shap_matrix[aapl_slice]
pred_aapl = y_pred_full[aapl_slice]
dates_aapl = features_by_sym[INTERP_SYM].index

print(f'AAPL slice: {n_aapl} bars | SHAP matrix {shap_aapl.shape}')

---
## 1 — Feature importances

Three XGBoost importance metrics. Disagreement between them is informative:
- **Gain:** average loss reduction per split (most reliable proxy for predictive value)
- **Cover:** average number of observations per split
- **Frequency (weight):** total number of splits using the feature

In [ ]:
imp_types = ['gain', 'cover', 'weight']
imp_dfs = {}
for t in imp_types:
    scores = booster.get_score(importance_type=t)
    # XGBoost assigns f0, f1... internally when fit via sklearn interface (no DMatrix).
    # Map back to human-readable feature names by position.
    s = pd.Series(
        [scores.get(f'f{i}', 0.0) for i in range(len(feature_names))],
        index=feature_names,
    )
    imp_dfs[t] = s / s.sum() if s.sum() > 0 else s

gain_order = imp_dfs['gain'].sort_values(ascending=True).index

fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=True)
for ax, t, color in zip(axes, imp_types, ['#4C72B0', '#55A868', '#C44E52']):
    vals = imp_dfs[t].loc[gain_order]
    ax.barh(range(len(gain_order)), vals.values, color=color, alpha=0.8)
    ax.set_yticks(range(len(gain_order)))
    ax.set_yticklabels(list(gain_order), fontsize=9)
    ax.set_title(f'Importance: {t}', fontsize=12)
    ax.set_xlabel('Normalized score')

fig.suptitle('Feature importances — IS-only GBM (three views; disagreement is informative)',
             y=1.01)
plt.tight_layout()
plt.show()

print('Top 5 by gain:')
print(imp_dfs['gain'].sort_values(ascending=False).head(5).to_string())
print()
for a, b in [('gain', 'cover'), ('gain', 'weight'), ('cover', 'weight')]:
    r = imp_dfs[a].corr(imp_dfs[b], method='spearman')
    print(f'Spearman rank corr  {a} vs {b}: {r:.3f}')

---
## 2 — SHAP summary

Each dot = one observation. X = SHAP value (contribution to predicted return).
Color = feature value (blue = low, red = high). Sorted by mean |SHAP|.

Computed via XGBoost native `booster.predict(pred_contribs=True)` — no `shap` library required.

In [ ]:
np.random.seed(0)

mean_abs_shap = np.abs(shap_matrix).mean(axis=0)
sorted_feat_idx = np.argsort(mean_abs_shap)

n_pts = min(3000, X_full.shape[0])
sample_idx = np.random.choice(X_full.shape[0], n_pts, replace=False)

fig, ax = plt.subplots(figsize=(11, 9))
for plot_row, feat_idx in enumerate(sorted_feat_idx):
    shap_vals = shap_matrix[sample_idx, feat_idx]
    feat_vals = X_full[sample_idx, feat_idx]
    fmin, fmax = np.nanpercentile(feat_vals, [1, 99])
    feat_norm = np.clip((feat_vals - fmin) / (fmax - fmin + 1e-10), 0, 1)
    y_jitter = plot_row + np.random.uniform(-0.35, 0.35, n_pts)
    sc = ax.scatter(shap_vals, y_jitter, c=feat_norm, cmap='RdBu_r',
                    s=4, alpha=0.35, vmin=0, vmax=1, linewidths=0)

ax.set_yticks(range(len(feature_names)))
ax.set_yticklabels([feature_names[i] for i in sorted_feat_idx], fontsize=9)
ax.axvline(0, color='black', linewidth=0.9, linestyle='--')
ax.set_xlabel('SHAP value (impact on predicted return)')
ax.set_title('SHAP summary — global feature impact\n(blue = low value, red = high value)',
             fontsize=12)
plt.colorbar(sc, ax=ax, label='Feature value (low \u2192 high)', fraction=0.02, pad=0.02)
plt.tight_layout()
plt.show()

print('\nMean |SHAP| — top 5 features:')
shap_imp = pd.Series(mean_abs_shap, index=feature_names).sort_values(ascending=False)
print(shap_imp.head(5).to_string())

---
## 3 — SHAP dependence plots

How does SHAP value vary with feature level for the top 3 features?
- Positive slope = trend-following signal
- Negative slope = mean-reversion signal
- Non-linearity reveals regime-conditional behavior

Points colored by `ma200_ratio` (blue = bear regime, red = bull regime).

In [ ]:
top3_idx   = np.argsort(mean_abs_shap)[-3:][::-1]
top3_names = [feature_names[i] for i in top3_idx]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
regime_col = 'ma200_ratio'
regime_idx = feature_names.index(regime_col) if regime_col in feature_names else None

for ax, feat_idx, feat_name in zip(axes, top3_idx, top3_names):
    feat_vals = X_full[sample_idx, feat_idx]
    shap_vals = shap_matrix[sample_idx, feat_idx]

    if regime_idx is not None and feat_idx != regime_idx:
        r_vals = X_full[sample_idx, regime_idx]
        rmin, rmax = np.nanpercentile(r_vals, [1, 99])
        color_vals = np.clip((r_vals - rmin) / (rmax - rmin + 1e-10), 0, 1)
        cbar_label = 'ma200_ratio (bear\u2192bull)'
    else:
        fmin, fmax = np.nanpercentile(feat_vals, [1, 99])
        color_vals = np.clip((feat_vals - fmin) / (fmax - fmin + 1e-10), 0, 1)
        cbar_label = f'{feat_name} (low\u2192high)'

    sc = ax.scatter(feat_vals, shap_vals, c=color_vals, cmap='RdBu_r',
                    s=5, alpha=0.4, vmin=0, vmax=1, linewidths=0)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xlabel(feat_name)
    ax.set_ylabel('SHAP value')
    ax.set_title(f'{feat_name}\n(#{list(top3_idx).index(feat_idx)+1} by mean |SHAP|)')
    plt.colorbar(sc, ax=ax, label=cbar_label, fraction=0.04, pad=0.04)

fig.suptitle('SHAP dependence plots — top 3 features', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

---
## 4 — Prediction decomposition

Waterfall chart for 3 AAPL IS bars: one correct long, one correct short, one wrong.
Shows which features drove each prediction: base value + SHAP contributions = predicted return.

In [ ]:
def waterfall_ax(ax, shap_vals, feature_names, actual, predicted, title, top_n=10):
    abs_s   = np.abs(shap_vals)
    top_idx = np.argsort(abs_s)[-top_n:]
    other   = shap_vals[np.setdiff1d(np.arange(len(shap_vals)), top_idx)].sum()

    names = [feature_names[i] for i in top_idx]
    vals  = shap_vals[top_idx]
    if abs(other) > 1e-9:
        names = ['(others)'] + names
        vals  = np.concatenate([[other], vals])

    colors = ['#4C72B0' if v >= 0 else '#C44E52' for v in vals]
    ax.barh(range(len(vals)), vals, color=colors, alpha=0.85)
    ax.set_yticks(range(len(vals)))
    ax.set_yticklabels(names, fontsize=8)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('SHAP contribution')
    tag = '\u2b06 CORRECT LONG'  if predicted > 0 and actual > 0 else \
          '\u2b07 CORRECT SHORT' if predicted < 0 and actual < 0 else \
          '\u2717 WRONG'
    ax.set_title(f'{title}\npred={predicted:+.4f}  actual={actual:+.4f}  {tag}', fontsize=9)

correct_long  = (pred_aapl > 0) & (y_aapl > 0)
correct_short = (pred_aapl < 0) & (y_aapl < 0)
wrong         = np.sign(pred_aapl) != np.sign(y_aapl)

def best_bar(mask):
    cands = np.where(mask)[0]
    return cands[np.argmax(np.abs(pred_aapl[cands]))] if len(cands) else 0

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, bar_idx, label in zip(axes,
    [best_bar(correct_long), best_bar(correct_short), best_bar(wrong)],
    ['Correct long', 'Correct short', 'Wrong']):
    waterfall_ax(ax, shap_aapl[bar_idx], feature_names,
                 y_aapl[bar_idx], pred_aapl[bar_idx],
                 f'{label} — AAPL {dates_aapl[bar_idx].date()}')

fig.suptitle('Prediction decomposition — AAPL IS bars (SHAP contributions)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

---
## 5 — Signal vs outcome scatter

Predicted return vs actual return for AAPL IS data. Green = correct direction, red = wrong.
OLS slope near 1 with low scatter = well-calibrated magnitude; flat slope with high hit rate
= correct direction but miscalibrated magnitude.

In [ ]:
correct_dir = np.sign(pred_aapl) == np.sign(y_aapl)
c_scatter   = ['#55A868' if c else '#C44E52' for c in correct_dir]

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(pred_aapl, y_aapl, c=c_scatter, s=8, alpha=0.4, linewidths=0)

m, b = np.polyfit(pred_aapl, y_aapl, 1)
xr = np.linspace(pred_aapl.min(), pred_aapl.max(), 100)
ax.plot(xr, m * xr + b, 'k--', linewidth=1.2, label=f'OLS slope={m:.2f}')
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)

hit_rate = correct_dir.mean()
ax.set_xlabel('Predicted return (IS-only GBM)')
ax.set_ylabel('Actual forward return')
ax.set_title(f'AAPL IS — Signal vs outcome  |  hit rate={hit_rate:.1%}  OLS slope={m:.2f}')
ax.legend(handles=[
    Patch(color='#55A868', alpha=0.6, label=f'Correct direction ({hit_rate:.1%})'),
    Patch(color='#C44E52', alpha=0.6, label=f'Wrong direction ({1-hit_rate:.1%})'),
], fontsize=9)
plt.tight_layout()
plt.show()

---
## 6 — Rolling directional accuracy

63-bar rolling hit rate over the AAPL IS period. Persistent runs above/below 50%
reveal regime sensitivity — the model may learn different things in different market phases.

In [ ]:
ROLL_W = 63
roll_hit = pd.Series(correct_dir.astype(float), index=dates_aapl).rolling(ROLL_W).mean()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(roll_hit.index, roll_hit * 100, linewidth=1.2, color='#4C72B0')
ax.axhline(50, color='black', linewidth=0.8, linestyle='--', label='50% (no skill)')
ax.fill_between(roll_hit.index, 50, roll_hit * 100,
                where=(roll_hit > 0.5), color='#55A868', alpha=0.25, label='Above 50%')
ax.fill_between(roll_hit.index, 50, roll_hit * 100,
                where=(roll_hit < 0.5), color='#C44E52', alpha=0.25, label='Below 50%')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.set_ylabel('Hit rate (%)')
ax.set_title(f'AAPL IS — {ROLL_W}-bar rolling directional accuracy (IS-only GBM)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f'Overall IS hit rate: {correct_dir.mean():.1%}')
print(f'Rolling range: [{roll_hit.dropna().min():.1%}, {roll_hit.dropna().max():.1%}]')

---
## 7 — Confusion matrix

Predicted direction vs actual direction across all 6 panel symbols (full IS matrix).
Row-normalized version answers: *When GBM predicts long/short, how often is it right?*

In [ ]:
nz_mask     = np.abs(y_pred_full) > 1e-9
pred_sign   = np.sign(y_pred_full[nz_mask]).astype(int)
actual_sign = np.sign(y_full[nz_mask]).astype(int)

labels = [+1, -1]
lnames = ['Long (+1)', 'Short (-1)']
cm = np.array([[((pred_sign == p) & (actual_sign == a)).sum() for a in labels] for p in labels])
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, data, title, fmt in [
    (axes[0], cm,      'Counts',          '{:,d}'),
    (axes[1], cm_norm, 'Row-normalized',  '{:.1%}'),
]:
    ax.imshow(data, cmap='Blues', vmin=0)
    ax.set_xticks([0, 1]); ax.set_xticklabels([f'Actual\n{n}' for n in lnames])
    ax.set_yticks([0, 1]); ax.set_yticklabels([f'Pred\n{n}' for n in lnames])
    ax.set_title(f'Confusion matrix — {title}')
    for pi in range(2):
        for ai in range(2):
            ax.text(ai, pi, fmt.format(data[pi, ai]), ha='center', va='center',
                    fontsize=12, color='white' if data[pi, ai] > data.max() * 0.6 else 'black')

plt.tight_layout()
plt.show()

print(f'When GBM predicts LONG:  correct {cm_norm[0, 0]:.1%}')
print(f'When GBM predicts SHORT: correct {cm_norm[1, 1]:.1%}')

---
## 8 — Ridge coefficients

Ridge is directly interpretable: each coefficient is the marginal expected-return
contribution per standardized unit of that feature. Error bars = 95% bootstrap CI
from 50 resamples. Wide CIs indicate unstable features; narrow CIs indicate robust ones.

In [ ]:
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_full)

ridge_is = Ridge(alpha=1.0)
ridge_is.fit(X_scaled, y_full)
coef_pt = ridge_is.coef_

np.random.seed(42)
n = len(y_full)
boot_coefs = np.zeros((50, len(feature_names)))
for i in range(50):
    idx = np.random.choice(n, n, replace=True)
    r = Ridge(alpha=1.0)
    r.fit(X_scaled[idx], y_full[idx])
    boot_coefs[i] = r.coef_

ci_lo = np.percentile(boot_coefs, 2.5, axis=0)
ci_hi = np.percentile(boot_coefs, 97.5, axis=0)

sort_order   = np.argsort(np.abs(coef_pt))
names_sorted = [feature_names[i] for i in sort_order]
coef_sorted  = coef_pt[sort_order]
ci_lo_s      = ci_lo[sort_order]
ci_hi_s      = ci_hi[sort_order]

fig, ax = plt.subplots(figsize=(9, 7))
y_pos   = np.arange(len(names_sorted))
colors  = ['#4C72B0' if c >= 0 else '#C44E52' for c in coef_sorted]
ax.barh(y_pos, coef_sorted, color=colors, alpha=0.75)
ax.errorbar(coef_sorted, y_pos,
            xerr=[coef_sorted - ci_lo_s, ci_hi_s - coef_sorted],
            fmt='none', color='black', capsize=3, linewidth=1.0, alpha=0.7)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(names_sorted, fontsize=9)
ax.set_xlabel('Standardized coefficient (95% bootstrap CI, 50 resamples)')
ax.set_title('Ridge IS coefficients — sorted by |coef|\n'
             '(blue = long signal, red = short signal)', fontsize=11)
plt.tight_layout()
plt.show()

print('\nTop 5 Ridge features by |coefficient|:')
coef_s = pd.Series(coef_pt, index=feature_names)
print(coef_s.abs().sort_values(ascending=False).head(5).to_string())